# Ungraded Lab: Saliency (PyTorch)

Like class activation maps, saliency maps also tell us what parts of the image the model is focusing on when making its predictions.
- The main difference is that in saliency maps, we are just shown the relevant pixels instead of the learned features.
- You can generate saliency maps by getting the gradient of the loss with respect to the image pixels.
- This means that changes in certain pixels that strongly affect the loss will be shown brightly in your saliency map.

Let's see how this is implemented in the following sections.

> This notebook is a PyTorch port of the original TensorFlow Hub lab. The Inception V3 classifier comes from torchvision instead of TF Hub, and the gradient is taken with `torch.autograd.grad` rather than a `GradientTape`.

## Imports

In [ ]:
import os
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torchvision.models import inception_v3, Inception_V3_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Build the model

For the classifier, you will use the [Inception V3 model](https://arxiv.org/abs/1512.00567) available in [torchvision](https://pytorch.org/vision/stable/models/generated/torchvision.models.inception_v3.html). This has pretrained weights that can detect 1000 ImageNet classes.

Two differences from the TensorFlow version are worth knowing:

- The TF Hub model predicted **1001** classes, because it keeps an extra "background" class at index 0. torchvision predicts **1000**, so every ImageNet class id here is one lower than the TF Hub equivalent.
- The torchvision weights expect inputs normalized with the ImageNet mean and standard deviation. We wrap that normalization in a layer *inside* the model, so the image you feed in stays in the `[0, 1]` range. That matters here, because the saliency map is the gradient with respect to whatever you feed in, and you want it to line up with the pixels you can actually see.

In [ ]:
class Normalize(nn.Module):
    '''Applies the ImageNet channel mean and standard deviation as a layer.'''

    def __init__(self):
        '''Registers the ImageNet channel mean and standard deviation as buffers.'''
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        '''
        Applies the ImageNet normalization to a batch of images.

        Args:
          x (tensor) -- batch of images in [0, 1], shape (N, 3, H, W)

        Returns:
          tensor -- the batch normalized with the ImageNet statistics
        '''
        return (x - self.mean) / self.std


# grab the pretrained classifier and put the normalization in front of it.
# the softmax is applied explicitly below rather than as a layer, which is the PyTorch convention.
weights = Inception_V3_Weights.IMAGENET1K_V1
class_names = weights.meta["categories"]

model = nn.Sequential(
    Normalize(),
    inception_v3(weights=weights),
).to(device).eval()

print(f"{len(class_names)} classes")

## Get a sample image

You will download a photo of a Siberian Husky that our model will classify. We left the option to download a Tabby Cat image instead if you want.

_(The file name of the husky photo says "pacific-ocean", but it really is a sled dog picture. That is just how it is named on the host.)_

In [ ]:
image_url = "https://cdn.pixabay.com/photo/2018/02/27/14/11/the-pacific-ocean-3185553_960_720.jpg"

# If you want to try the cat, uncomment this line
# image_url = "https://cdn.pixabay.com/photo/2017/02/20/18/03/cat-2083492_960_720.jpg"

if not os.path.exists("image.jpg"):
    request = urllib.request.Request(image_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, open("image.jpg", "wb") as out:
        out.write(response.read())
print("saved image.jpg")

## Preprocess the image

The image needs to be preprocessed before being fed to the model. This is done in the following steps:

In [ ]:
# read the image
img = cv2.imread('image.jpg')

# format it to be in the RGB colorspace
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# resize to 299x299, the size Inception V3 was trained at, and normalize pixel values to [0, 1]
img = cv2.resize(img, (299, 299)) / 255.0

# move the channels first and add a batch dimension: (1, 3, 299, 299)
image = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0)

We can now preview our input image.

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

Before computing the gradients, it is worth checking what the model actually predicts for this image.

In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(model(image.to(device)), dim=1)[0]

top = probabilities.topk(3)
for score, idx in zip(top.values, top.indices):
    print(f"  {class_names[idx]:20s} {float(score):.3f}   (class index {int(idx)})")

## Compute Gradients

You will now get the gradients of the loss with respect to the input image pixels. This is the key step to generate the map later.

Note the class index. In the TensorFlow version the Siberian Husky was class `251`, because TF Hub's model has an extra background class at index 0. Here the same class is `250`.

In [ ]:
# Siberian Husky's class ID in torchvision's ImageNet ordering (251 in the TF Hub model)
class_index = 250

# If you downloaded the cat, use this line instead
# class_index = 281   # Tabby Cat (282 in the TF Hub model)

# number of classes the model predicts
num_classes = len(class_names)

# convert to a one hot representation to match the softmax below
expected_output = torch.nn.functional.one_hot(
    torch.tensor([class_index] * image.shape[0]), num_classes
).float().to(device)

# cast the image to float and put it on the device
inputs = image.clone().to(device)

# watch the input pixels, so autograd tracks the gradient with respect to them
inputs.requires_grad_(True)

# generate the predictions
predictions = torch.softmax(model(inputs), dim=1)

# get the loss, the categorical cross entropy between the one hot target and the predictions
loss = -(expected_output * torch.log(predictions + 1e-10)).sum(dim=1)

# get the gradient with respect to the inputs
gradients = torch.autograd.grad(loss.sum(), inputs)[0]

print("gradients shape:", tuple(gradients.shape))

## Visualize the results

Now that you have the gradients, you will do some postprocessing to generate the saliency maps and overlay them on the image.

In [ ]:
# reduce the RGB image to grayscale by summing the absolute gradient over the channel axis
grayscale_tensor = gradients.abs().sum(dim=1)

# normalize the pixel values to be in the range [0, 255].
# the max value in the grayscale tensor will be pushed to 255.
# the min value will be pushed to 0.
normalized_tensor = (
    255
    * (grayscale_tensor - grayscale_tensor.min())
    / (grayscale_tensor.max() - grayscale_tensor.min())
).to(torch.uint8)

# remove the batch dimension to make the tensor a 2d tensor
normalized_tensor = normalized_tensor.squeeze().cpu()

Let's do a little sanity check to see the results of the conversion.

In [ ]:
grayscale = grayscale_tensor[0].detach().cpu().numpy()

# max and min value in the grayscale tensor
print(np.max(grayscale))
print(np.min(grayscale))
print()

# coordinates of the first pixel where the max and min values are located
max_pixel = np.unravel_index(np.argmax(grayscale), grayscale.shape)
min_pixel = np.unravel_index(np.argmin(grayscale), grayscale.shape)
print(max_pixel)
print(min_pixel)
print()

# these coordinates should have the max (255) and min (0) value in the normalized tensor
print(normalized_tensor[max_pixel])
print(normalized_tensor[min_pixel])

The exact numbers will differ from the TensorFlow version, since this is a different copy of Inception V3 with different weights, but the two printed pixels should always hold 255 and 0:

```
tensor(255, dtype=torch.uint8)
tensor(0, dtype=torch.uint8)
```

Now let's see what this looks like when plotted. The white pixels show the parts the model focused on when classifying the image.

In [ ]:
plt.figure(figsize=(8, 8))
plt.axis('off')
plt.imshow(normalized_tensor, cmap='gray')
plt.show()

Let's superimpose the normalized tensor on the input image to get more context. You can see that the strong pixels sit over the dog, which is a good indication that the model is looking at the correct part of the image.

In [ ]:
gradient_color = cv2.applyColorMap(normalized_tensor.numpy(), cv2.COLORMAP_HOT)
gradient_color = gradient_color / 255.0
super_imposed = cv2.addWeighted(img, 0.5, gradient_color, 0.5, 0.0)

plt.figure(figsize=(8, 8))
plt.imshow(super_imposed)
plt.axis('off')
plt.show()